# Setup

In [ ]:
!pip install -U \
  langchain-core \
  langgraph \
  langchain-google-genai \
  google-generativeai \
  pymupdf \
  pillow \
  markdown \
  lxml \
  pandas \
  tqdm

In [ ]:
!unzip /content/drive/MyDrive/Project_Medical_LMM/Data_Processing/extracted_case_report_image_filtered.zip -d /content/

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types
from langchain_core.messages import HumanMessage
from typing import TypedDict, Optional, List
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from pathlib import Path
from IPython.display import Image, display

import os
import json
import re
import time
import pandas as pd
import pprint

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY_3")

In [ ]:
MODEL_2 = "gemma-3-27b-it"
CONFIG = types.GenerateContentConfig(
  temperature=0.1
)

# Utils

In [ ]:
def split_text_to_tag(text, tags):
  parsed_data = {}

  tag_pattern = '|'.join(re.escape(tag) for tag in tags)
  split_regex = fr'(<\/?(?:{tag_pattern})>)'

  split_result = re.split(split_regex, text)

  cleaned_split_result = [item.strip() for item in split_result if item.strip()]

  current_tag = None

  for item in cleaned_split_result:
    if item.startswith('<') and item.endswith('>'):
      if item.startswith('</'): # Closing tag
        current_tag = None
      else: # Opening tag
        current_tag = item[1:-1] # Extract tag name without <>
        parsed_data[current_tag] = "" # Initialize with empty string, will be populated by next item
    elif current_tag:
      # Append to existing content, as there might be multiple lines of text between tags
      if parsed_data[current_tag]:
        parsed_data[current_tag] += "\n" + item.strip()
      else:
        parsed_data[current_tag] = item.strip()
  return parsed_data

In [ ]:
def extract_image_paths(text):
  images = []
  pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
  matches = re.findall(pattern, text)

  if matches:
    images = [img_path for _, img_path in matches]

  return images

In [ ]:
from google.genai import types

def load_images_as_base64(images_dir: str):
  images = []
  for img_path in Path(images_dir).glob("*.jpg"):
    with open(img_path, "rb") as f:
      images.append(
        types.Part.from_bytes(
          data=f.read(),
          mime_type="image/jpeg"
        )
      )
  return images

In [ ]:
def load_case_report(case_path: str):
  case_path = Path(case_path)

  md_file = next(case_path.glob("*.md"))
  images_dir = case_path / "images"

  with open(md_file, "r") as f:
    text = f.read()

  images = load_images_as_base64(images_dir)
  return text, images

In [ ]:
def select_synthesized_fields(state: dict) -> dict:
  state["flagged"] = "Yes" if state.get("flags", "GRADE_FAILED") != "NONE" else "No"

  return {
    "source": state["source"],
    "case_prompt": state.get("generated_case_prompt"),
    "final_diagnosis": state.get("generated_final_diagnosis"),
    "reasoning_points": state.get("generated_reasoning_points"),
    "reasoning_narrative": state.get("generated_reasoning_narrative"),
    "flagged": state.get("flagged"),
  }

# Prompt

In [ ]:
def get_quality_grader_prompt(full_text):
  quality_grader_prompt = f"""
    You are an expert medical educator tasked with evaluating case reports for their diagnostic-reasoning value.
    You will be given full, uncleaned text that has just been extracted from PDF via OCR. You will have to understand the text because some parts are splited due to layout.
    These images are passed to you together with the case.
    The goal is to check if this case report can be use like a diagnostic teaching cases from medical textbooks.

    CASE REPORT EVALUATION RUBRIC
    >> HOW TO USE
    1. Read the entire case once without scoring.
    2. Re-read, taking notes.
    3. Inside <think>...</think>, write the reasoning that leads you to each score.
    4. Output only the five XML tags shown after the rubric—nothing else.

    +-------------------------------------------------------------+
    | 1. THOROUGHNESS OF CASE PRESENTATION (1-5 points) |
    | Look for: HPI, past history, meds, allergies, vitals,   |
    | focused exam, labs, imaging, hospital course, outcome.  |
    | 1: Seriously deficient (identifiers only; no vitals)    |
    | 2: Major gaps (HPI + vitals OR exam, not both).         |
    | 3: Adequate (present but sketchy details).              |
    | 4: Very good (complete data, clear timeline).           |
    | 5: Exemplary (serial data & course, high quality).      |
    +-------------------------------------------------------------+
    | 2. EXPLICIT DIFFERENTIAL DIAGNOSIS (Yes / No)             |
    | >=2 plausible alternatives? If yes → "Yes"; else → "No".  |
    +-------------------------------------------------------------+
    | 3. DEPENDENCE ON INTEGRATIVE CLINICAL REASONING (1-5)     |
    | Measures need to combine >=2 data points (hx, labs, etc)  |
    | 1: Trivial: lone clue gives answer.                       |
    | 2: Minimal: one dominant clue.                            |
    | 3: Moderate: must merge TWO findings.                     |
    | 4: High: THREE+ clues; requires synthesis.                |
    | 5: Outstanding: stepwise, complex reasoning.              |
    +-------------------------------------------------------------+
    | 4. TRANSPARENCY OF DIAGNOSTIC REASONING PROCESS (1-5) |
    | 1: None (no rationale).                               |
    | 2: Superficial (lists w/o "why").                     |
    | 3: Adequate (brief pivots).                           |
    | 4: Detailed (stepwise, probabilities).                |
    | 5: Model (structured, addresses pitfalls).            |
    +-------------------------------------------------------------+
    | 5. USEFULNESS OF IMAGES (1-5)                                  |
    | 1: None (no usefulness or relatedness).                        |
    | 2: Low (very limited or marginal usefulness or relevance).     |
    | 3: Average (illustration or example, limited relevance).       |
    | 4: High (high usefulness and relevance, but can be omitted).   |
    | 5: Very high (cannot be omitted without serious consequences). |
    +-------------------------------------------------------------+
    | 5. STATED FINAL DIAGNOSIS (Yes / No)                  |
    | Is diagnosis clearly named? Yes → "Yes"; else → "No". |
    +-------------------------------------------------------------+

    Additional Rules:
    1. If the given article is not actually a case report, output ”NA” for all scores.
    2. Think through whether the final diagnosis can be reasonably deduced from the case presentation when determining the
    educational value in #3.

    OUTPUT TEMPLATE (leave tags exactly as written)
    <think>
    ...your internal reasoning for each item...
    </think>

    <case_presentation_score>[1-5]</case_presentation_score>
    <differential_diagnosis_score>[Yes/No]</differential_diagnosis_score>
    <integrative_reasoning_score>[1-5]</integrative_reasoning_score>
    <transparency_score>[1-5]</transparency_score>
    <images_usefulness_score>[1-5]</images_usefulness_score>
    <final_diagnosis_score>[Yes/No]</final_diagnosis_score>

    SUPPLIED CASE REPORT
    <full_text>
    {full_text}
    </full_text>
  """

  return quality_grader_prompt

In [ ]:
def get_extractor_prompt(full_text):
  extractor_prompt = f"""
    You are an expert clinician–educator. You are given a journal diagnostic case. Your main job is to:
    - Further extract factual details from the full text of the case. The text hasn't been cleaned or augmented.

    Besides that, your job is also to:
    - Summarize the key information of the patient for diagnosis.
    - Summarize the differential diagnosis process, including the rationale for each step and the reasons for considering or excluding specific diagnoses.
    - Summarize the final diagnosis of the patient.
    - Understand the text because some parts are splited due to the layout.
    - Check for formatting errors and typo and fix them.

    The case includes the image path after each figure. These images are passed to you together with the case.

    Ensure that your summaries are concise and accurate, based solely on the information provided in the case report.
    If the case report is incomplete or does not meet the requirements for summarization, simply output: 'I can't.'

    RULES (Read Carefully—No Exceptions)
    1. Source Fidelity – Extract facts only from the supplied case report.
    • Do NOT invent, embellish, or “smooth out” missing data.
    • Paraphrase narrative prose into concise bullets where helpful, but never add new facts.

    2. Structure the Teaching Case
    Case Presentation → Additional Infos → Question-Answer pairs → Follow-up → Disease summary and remarks

    3. Use the XML Tags Exactly as Shown
    • <think> . . . </think> – your hidden analytic notes (not visible to students).
    • <image_finding> . . . </image_finding> - your judgment about the meaning and content of the image and the helpfulness of the image.
    • <case_prompt> . . . </case_prompt> – the information given to students before they generate a differential.
    • <reasoning_points> . . . </reasoning_points> – numbered bullet reasons, each built as a full sentence followed by a direct quote.
    • <reasoning_narrative> . . . </reasoning_narrative> - the continuous narrative of the reasoning points.
    • <final_diagnosis> . . . </final_diagnosis> – single disease/entity name only, nothing more.

    4. What Goes Inside <think>
    • Key points – What makes this case non-trivial or pedagogically interesting? This should guide where the breakpoint should be.
    • Ideal breakpoint – What details of the case presentation should you include and exclude so that students have enough data to reason, but
    no spoilers?
    • Author’s analytic distinctions – How did they reach and separate the final diagnosis from look-alikes and other conditions?

    5. What Goes Inside <image_finding>
    • Description - What the image shows or displays
    • Helpfulness - How the image helps or supports the doctor in understanding the case.
    • Relevance - How relevant the image is to the patient.

    6. What Goes Inside <case_prompt>
    • Present only the facts known when recieve the patient and before a working differential was made: chief complaint, HPI, vitals, physical exam, and early investigations. No future updates or follow-up.
    • Perserve the lab and examination results.
    • Include images' names that related closely with the case, within the text right after the fact ( for example, this patient has some X-ray (Fig 100.100) ). Ensure that those images are necessary for the case prompt.
    • Present the case in the order presented in the case report (e.g., physical labs before imaging, etc.).
    • Omit any wording that directly states or hints at the final diagnosis.
    • Present this as closely as possible to the style in which the case report is written.
    • Omit repeating details.
    • Omit the answer to the question “What is the final diagnosis?”
    • Include only one question. Do not include answer.
    • Put everything into a paragraph.

    7. What Goes Inside <reasoning_points>
    • Numbered list (1., 2., 3., . . . ).
    • Discuss only about the final diagnosis, not any other findings.
    • Each entry: concise summary of reason [“direct quote from article”]. You can use ellipses (. . . ) to shorten the quote if there are irrelevant details.
    • The basis for the diagnosis and highlight the key factors supporting this conclusion. Use only the information from the case prompt
    • The steps to reach the final diagnosis from the case prompt.
    • Discuss and explain the steps to reach the final diagnosis from the case prompt.

    8. What Goes Inside <reasoning_narrative>
    • Stitching the reasoning points into a continous narrative.
    • Structure: Diagnostic steps -> Image grounding.

    9. What Goes Inside <final_diagnosis>
    • Single disease/entity name (e.g., sarcoidosis).
    • No adjectives, punctuation, or explanatory text.
    • Answer the case prompt.

    OUTPUT TEMPLATE (copy exactly, especially the tag)
    <think>
    1. [Core tension]
    2. [Best breakpoint of case report, what to include and what to exclude]
    3. [Key analytic distinctions between competing diagnoses (taken from case report)]
    </think>

    <image_finding>
    1. image_1
    Description: ...
    Helpfulness: ...
    Relevance: ...

    2. image_2
    Description: ...
    Helpfulness: ...
    Relevance: ...

    3. ...
    </image_finding>

    <case_prompt>
    [Your case presentation text, faithful to the report and stopping at the breakpoint]
    </case_prompt>

    <reasoning_points>
    1. reasoning_point_1 | Direct quote from article: "..." | Grounded in image if possible: ...

    2. reasoning_point_2 | Direct quote from article: "..." | Grounded in image if possible: ...

    3. ...
    </reasoning_points>

    <reasoning_narrative>
    ReasoningNarrative
    </reasoning_narrative>

    <final_diagnosis>
    DiseaseName
    </final_diagnosis>

    SUPPLIED CASE REPORT
    <full text>
    {full_text}
    </full text>
  """

  return extractor_prompt

In [ ]:
def get_editor_prompt(extractor_prompt, full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis):
  editor_prompt = f"""
    You are the Editor.
    Your job is to audit a draft teaching case that was produced from a published case report.
    Image are passed to you together with the case.

    You must confirm strict compliance with all instructions, detect hallucinations, and ensure pedagogic quality.

    YOUR INPUTS
      1. The original draft you must audit appears between <generated case> . . . </generated case>.
      2. The source article appears between <case report> . . . </case report>.

      Here is the original guideline the draft was produced in accordance with:
        {extractor_prompt}
      Don't take this guideline in this part as your instruction. This is just for you to review.

    CHECKLIST — FAIL ANY ITEM → RAISE A FLAG
      A. Source Fidelity
        □ Every fact in each section is traceable to the source article.
        □ No invented details or embellishments.
      B. Case Presentation Quality
        □ All facts from the case prompt are present in the source article.
        □ Contains only information known before the clinicians formed a differential.
        □ Images included must relate closely to the case prompt
        □ Does not reveal the final diagnosis (there should be room for at least some inference).
        □ Provides sufficient data (HPI, vitals, exam ± initial tests) for clinicians to formulate a reasonable differential and get the correct final diagnosis.
      C. Diagnostic Reasoning Section
        □ Each numbered entry starts with a summary of the reasoning plus a direct quote from the article.
        □ Quotes are verbatim or use ellipses (. . . ) without changing meaning.
        □ Paraphrased quotes are okay, as long as they retain the original meaning.
        □ Rationales reference only information that already appears in <case prompt> (not based on new findings, confirmatory tests, or data withheld from students).
      D. Final Diagnosis Tag
        □ Final diagnosis is reasonably deducible from the case-presentation facts. — i.e., the final diagnosis should not depend entirely on some test, imaging, or lab result not given in the case presentation.
      E. No Hallucinations Anywhere
        □ Every datum, quote, or diagnosis is found in the case report.

    HOW TO REPORT YOUR FINDINGS
      Output only the two XML blocks below.
        1. <flags> . . . </flags>
          • If an item fails, add a line FLAG: [short descriptor].
          • Use one line per failed item, drawn from this controlled vocabulary:
            CASE PROMPT HALLUCINATION, FINAL DIAGNOSIS IN CASE PROMPT,
            INSUFFICIENT INFO FOR DIAGNOSIS, DIAGNOSTIC REASONING HALLUCINATION, OTHER.
          • If no issues, write NONE.
        2. <editor comments> . . . </editor comments>
          • Briefly justify each flag (one sentence each).
          • If no flags, you may omit or leave empty.

      Example when problems exist:
        <flags>
          FLAG: SOURCE_FIDELITY
          FLAG: REASONING_EXTRA_INFO
        </flags>

        <editor_comments>
          SOURCE_FIDELITY: Mentions \family history of SLE," not present in article.
          REASONING_EXTRA_INFO: Rationale cites a biopsy result that is not included in the case_prompt.
        </editor_comments>

    OUTPUT WHEN EVERYTHING PASSES:
        <flags>
          NONE
        </flags>
        <editor_comments></editor_comments>

    INPUT BLOCKS TO REVIEW
      Here is the reference case report:
        <case_report>
          {full_text}
        </case_report>

      Here is the diagnostic case generated by the model:
        <think>
          {generated_think}
        </think>
        <image_finding>
          {generated_image_finding}
        </image_finding>
        <case_prompt>
          {generated_case_prompt}
        </case_prompt>
        <reasoning_points>
          {generated_reasoning_points}
        </reasoning_points>
        <reasoning_narrative>
          {generated_reasoning_narrative}
        </reasoning_narrative>
        <final_diagnosis>
          {generated_final_diagnosis}
        </final_diagnosis>
  """

  return editor_prompt

In [ ]:
def get_retry_extractor_prompt(full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis, flags, editor_comments):
  retry_extractor_prompt = f"""
    You are an expert clinician–educator. You are given a journal diagnostic case, your previous extraction of the case and the feedback from the editor. Your main job is to:
    - Revise your extraction given the feedback from the editor.

    Besides that, your job is also to:
    - Further extract factual details from the full text of a journal diagnostic case. The text hasn't been cleaned or augmented.
    - Summarize the key information of the patient for diagnosis.
    - Summarize the differential diagnosis process, including the rationale for each step and the reasons for considering or excluding specific diagnoses.
    - Summarize the final diagnosis of the patient.
    - Understand the text because some parts are splited due to the layout.
    - Check for formatting errors and typo and fix them.

    The case includes the image path after each figure. These images are passed to you together with the case.

    Ensure that your summaries are concise and accurate, based solely on the information provided in the case report.
    If the case report is incomplete or does not meet the requirements for summarization, simply output: 'I can't.'

    RULES (Read Carefully—No Exceptions)
    1. Source Fidelity – Extract facts only from the supplied case report.
    • Do NOT invent, embellish, or “smooth out” missing data.
    • Paraphrase narrative prose into concise bullets where helpful, but never add new facts.

    2. Structure the Teaching Case
    Case Presentation → Additional Infos → Question-Answer pairs → Follow-up → Disease summary and remarks

    3. Use the XML Tags Exactly as Shown
    • <think> . . . </think> – your hidden analytic notes (not visible to students).
    • <image_finding> . . . </image_finding> - your judgment about the meaning and content of the image and the helpfulness of the image.
    • <case_prompt> . . . </case_prompt> – the information given to students before they generate a differential.
    • <reasoning_points> . . . </reasoning_points> – numbered bullet reasons, each built as a full sentence followed by a direct quote.
    • <reasoning_narrative> . . . </reasoning_narrative> - the continuous narrative of the reasoning points.
    • <final_diagnosis> . . . </final_diagnosis> – single disease/entity name only, nothing more.

    4. What Goes Inside <think>
    * Key points – What makes this case non-trivial or pedagogically interesting? This should guide where the breakpoint should be.
    * Ideal breakpoint – What details of the case presentation should you include and exclude so that students have enough data to reason, but
    no spoilers?
    * Author’s analytic distinctions – How did they reach and separate the final diagnosis from look-alikes and other conditions?

    5. What Goes Inside <image_finding>
    • Description - What the image shows or displays
    • Helpfulness - How the image helps or supports the doctor in understanding the case.
    • Relevance - How relevant the image is to the patient.

    6. What Goes Inside <case_prompt>
    • Present only the facts known when recieve the patient and before a working differential was made: chief complaint, HPI, vitals, physical exam, and early investigations. No future updates or follow-up.
    • Perserve the lab and examination results.
    • Include images' names that related closely with the case, within the text right after the fact ( for example, this patient has some X-ray (Fig 100.100) ). Ensure that those images are necessary for the case prompt.
    • Present the case in the order presented in the case report (e.g., physical labs before imaging, etc.).
    • Omit any wording that directly states or hints at the final diagnosis.
    • Present this as closely as possible to the style in which the case report is written.
    • Omit repeating details.
    • Omit the answer to the question “What is the final diagnosis?”
    • Include only one question. Do not include answer.
    • Put everything into a paragraph.

    7. What Goes Inside <reasoning_points>
    • Numbered list (1., 2., 3., . . . ).
    • Discuss only about the final diagnosis, not any other findings.
    • Each entry: concise summary of reason [“direct quote from article”]. You can use ellipses (. . . ) to shorten the quote if there are irrelevant details.
    • The basis for the diagnosis and highlight the key factors supporting this conclusion. Use only the information from the case prompt
    • The steps to reach the final diagnosis from the case prompt.
    • Discuss and explain the steps to reach the final diagnosis from the case prompt.

    8. What Goes Inside <reasoning_narrative>
    • Stitching the reasoning points into a continous narrative.
    • Structure: Diagnostic steps -> Image grounding.

    9. What Goes Inside <final_diagnosis>
    • Single disease/entity name (e.g., sarcoidosis).
    • No adjectives, punctuation, or explanatory text.
    • Answer the case prompt.

    OUTPUT TEMPLATE (copy exactly, especially the tag)
    <think>
    1. [Core tension]
    2. [Best breakpoint of case report, what to include and what to exclude]
    3. [Key analytic distinctions between competing diagnoses (taken from case report)]
    </think>

    <image_finding>
    1. image_1
    Description: ...
    Helpfulness: ...
    Relevance: ...

    2. image_2
    Description: ...
    Helpfulness: ...
    Relevance: ...

    3. ...
    </image_finding>

    <case_prompt>
    [Your case presentation text, faithful to the report and stopping at the breakpoint]
    </case_prompt>

    <reasoning_points>
    1. reasoning_point_1 | Direct quote from article: "..." | Grounded in image if possible: ...

    2. reasoning_point_2 | Direct quote from article: "..." | Grounded in image if possible: ...

    3. ...
    </reasoning_points>

    <reasoning_narrative>
    ReasoningNarrative
    </reasoning_narrative>

    <final_diagnosis>
    DiseaseName
    </final_diagnosis>


    SUPPLIED CASE REPORT
    <full text>
    {full_text}
    </full text>

    PREVIOUS EXTRACTION
    <think>
      {generated_think}
    </think>
    <image_finding>
      {generated_image_finding}
    </image_finding>
    <case_prompt>
      {generated_case_prompt}
    </case_prompt>
    <reasoning_points>
      {generated_reasoning_points}
    </reasoning_points>
    <reasoning_narrative>
      {generated_reasoning_narrative}
    </reasoning_narrative>
    <final_diagnosis>
      {generated_final_diagnosis}
    </final_diagnosis>

    EDITOR COMMENTS
    <flags>
      {flags}
    </flags>
    <editor_comments>
      {editor_comments}
    </editor_comments>
  """

  return retry_extractor_prompt

# State and Graph

In [ ]:
class CaseState(TypedDict):
  source: str
  case_path: str

  full_text: str
  images: list

  # grading
  grading: Optional[dict]
  passed_grading: bool

  # monitor
  retry: bool
  loop_count: int

  # extraction
  generated_think: Optional[str]
  generated_image_finding: Optional[str]
  generated_case_prompt: Optional[str]
  generated_reasoning_points: Optional[str]
  generated_reasoning_narrative: Optional[str]
  generated_final_diagnosis: Optional[str]

  flags: Optional[str]
  editor_comments: Optional[str]

In [ ]:
multimodal_llm = ChatGoogleGenerativeAI(
  model=MODEL_2,
  google_api_key=os.environ["GEMINI_API_KEY"],
  temperature=0.1,
)

agent = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
def load_case(state: CaseState):
    full_text, images = load_case_report(state["case_path"])

    time.sleep(5)

    return {
        "full_text": full_text,
        "images": images,
        "retry": False,
        "loop_count": 0,
    }

In [ ]:
def grade_case(state: CaseState):
    prompt = get_quality_grader_prompt(state["full_text"])

    contents = [prompt, *state["images"]]

    response = agent.models.generate_content(
        model=MODEL_2,
        contents=contents,
        config=CONFIG,
    )

    if response.text in (None, "I can't."):
        return {"passed_grading": False}

    grading = split_text_to_tag(
        response.text,
        [
            "think",
            "case_presentation_score",
            "differential_diagnosis_score",
            "integrative_reasoning_score",
            "transparency_score",
            "images_usefulness_score",
            "final_diagnosis_score",
        ],
    )

    passed = (
        int(grading["case_presentation_score"]) >= 3
        and int(grading["integrative_reasoning_score"]) >= 3
        and int(grading["transparency_score"]) >= 3
        and int(grading["images_usefulness_score"]) >= 3
        and grading["differential_diagnosis_score"] == "Yes"
        and grading["final_diagnosis_score"] == "Yes"
    )

    grading["source"] = state["source"]

    time.sleep(5)

    return {
        "grading": grading,
        "passed_grading": passed,
    }

In [ ]:
def grade_route(state: CaseState):
  if state["passed_grading"]:
    time.sleep(5)
    return "extract_case"

  time.sleep(5)
  return END

In [ ]:
def extract_case(state: CaseState):
    full_text = state["full_text"]

    if state["retry"]:
        prompt = get_retry_extractor_prompt(
            full_text,
            state["generated_think"],
            state["generated_image_finding"],
            state["generated_case_prompt"],
            state["generated_reasoning_points"],
            state["generated_reasoning_narrative"],
            state["generated_final_diagnosis"],
            state["flags"],
            state["editor_comments"],
        )
    else:
        prompt = get_extractor_prompt(full_text)

    contents = [prompt, *state["images"]]

    response = agent.models.generate_content(
        model=MODEL_2,
        contents=contents,
        config=CONFIG,
    )

    if response.text in (None, "I can't."):
      time.sleep(5)
      return {
          "flags": "NONE",
          "generated_think": state.get("generated_think"),
          "generated_image_finding": state.get("generated_image_finding"),
          "generated_case_prompt": state.get("generated_case_prompt"),
          "generated_reasoning_points": state.get("generated_reasoning_points"),
          "generated_reasoning_narrative": state.get("generated_reasoning_narrative"),
          "generated_final_diagnosis": state.get("generated_final_diagnosis"),
      }

    parsed = split_text_to_tag(
        response.text,
        [
            "think",
            "image_finding",
            "case_prompt",
            "reasoning_points",
            "reasoning_narrative",
            "final_diagnosis",
        ],
    )

    time.sleep(5)
    return {
        "generated_think": parsed["think"],
        "generated_image_finding": parsed["image_finding"],
        "generated_case_prompt": parsed["case_prompt"],
        "generated_reasoning_points": parsed["reasoning_points"],
        "generated_reasoning_narrative": parsed["reasoning_narrative"],
        "generated_final_diagnosis": parsed["final_diagnosis"],
    }

In [ ]:
def review_case(state: CaseState):
    editor_prompt = get_editor_prompt(
        "",
        state["full_text"],
        state["generated_think"],
        state["generated_image_finding"],
        state["generated_case_prompt"],
        state["generated_reasoning_points"],
        state["generated_reasoning_narrative"],
        state["generated_final_diagnosis"],
    )

    contents = [editor_prompt, *state["images"]]

    response = agent.models.generate_content(
        model=MODEL_2,
        contents=contents,
        config=CONFIG,
    )

    parsed = split_text_to_tag(response.text, ["flags", "editor_comments"])

    time.sleep(5)
    return {
        "flags": parsed["flags"],
        "editor_comments": parsed["editor_comments"],
        "retry": parsed["flags"] != "NONE",
        "loop_count": state["loop_count"] + 1,
    }

In [ ]:
def review_route(state: CaseState):
    if state["flags"] == "NONE":
        return END
    return "extract_case"

In [ ]:
graph = StateGraph(CaseState)

graph.add_node("load_case", load_case)
graph.add_node("grade_case", grade_case)
graph.add_node("extract_case", extract_case)
graph.add_node("review_case", review_case)

graph.set_entry_point("load_case")

graph.add_edge("load_case", "grade_case")

graph.add_conditional_edges(
    "grade_case",
    grade_route,
    {
        "extract_case": "extract_case",
        END: END,
    },
)

graph.add_edge("extract_case", "review_case")

graph.add_conditional_edges(
    "review_case",
    review_route,
    {
        "extract_case": "extract_case",
        END: END,
    },
)

app = graph.compile()
display(Image(app.get_graph(xray=1).draw_mermaid_png()))

# Debug run on 1 case

In [ ]:
import os

result = app.invoke({
    "source": "case_4",
    "case_path": f"/content/extracted_case_report_image_filtered/case_4/auto",
})

record = select_synthesized_fields(result)

out_path = f"/content/outputs/{result['source']}.json"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, "w") as f:
    json.dump(record, f, indent=2)

In [ ]:
pprint.pprint(result['generated_case_prompt'])

In [ ]:
pprint.pprint(result['generated_reasoning_points'])

In [ ]:
pprint.pprint(result['generated_reasoning_narrative'])

In [ ]:
pprint.pprint(result['generated_final_diagnosis'])

In [ ]:
display(result)

# Run on all cases

In [ ]:
from pathlib import Path
import json

CASES_ROOT = Path("/content/extracted_case_report_image_filtered/")
OUTPUT_DIR = Path("/content/outputs/")
OUTPUT_DIR.mkdir(exist_ok=True)

for case_dir in CASES_ROOT.iterdir():
    if not case_dir.is_dir():
        continue

    source = case_dir.name
    case_path = case_dir / "auto"

    out_file = OUTPUT_DIR / f"{source}.json"
    if out_file.exists():
        print(f"⏭ {source} already exists, skipping")
        continue

    print(f"▶ Processing {source}")

    try:
        result = app.invoke({
            "source": source,
            "case_path": str(case_path),
        })

        record = select_synthesized_fields(result)

        with open(OUTPUT_DIR / f"{source}.json", "w") as f:
            json.dump(record, f, indent=2)

    except Exception as e:
        print(f"❌ Failed {source}: {e}")

    print(f"✅ Done {source}")
    print()
    time.sleep(15)

In [ ]:
!zip -r /content/outputs.zip /content/outputs

In [ ]:
from google.colab import files
files.download('/content/outputs.zip')

# Save to CSV

In [ ]:
import pandas as pd
import json
from pathlib import Path

output_dir = Path("/content/outputs/")
all_records = []

for json_file in output_dir.glob("*.json"):
    with open(json_file, "r") as f:
        record = json.load(f)
        all_records.append(record)

df = pd.DataFrame(all_records)

df.to_csv("/content/moodle_93_cases.csv", index=False)

# Qualitative exploration

In [ ]:
import pandas as pd
import pprint

In [ ]:
data = pd.read_csv("/content/moodle_93_cases.csv")

In [ ]:
data = data.dropna()

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
pprint.pprint(data.iloc[2]['case_prompt'])

In [ ]:
for i in range(len(data)):
  source = data.iloc[i]['source']

  data_filter = data[data['source'] == source]

  print(f"{source}'s case prompt appear {len(data_filter)} times")

In [ ]:
print(f"Number of unique sources: {data['case_prompt'].nunique()}")
print(f"Total number of rows: {len(data)}")

In [ ]:
data.iloc[3]['source']

In [ ]:
pprint.pprint(data.iloc[2]['reasoning_narrative'])

# Adding image

In [ ]:
import shutil
import os

os.makedirs("/content/benchmark_image", exist_ok=True)

for i in range(len(data)):
  source = data.iloc[i]['source']
  question = data.iloc[i]['case_prompt']

  # Filter image-path pair
  with open(f"/content/extracted_case_report_image_filtered/{source}/auto/{source}.md") as f:
    content = f.read()

  image_path_pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
  image_path_pairs = list(set(re.findall(image_path_pattern, content))) # Remove duplicates

  # Filter figure mention
  figure_pattern = r'(Fig\.\s*\d+\.?\d*)'

  # Check if question is a string before applying re.findall
  if isinstance(question, str):
    figures = list(set(re.findall(figure_pattern, question)))
  else:
    figures = [] # Assign an empty list if question is not a string

  print(source)
  print(image_path_pairs)
  print(figures)

  # Include images
  if image_path_pairs and figures:
    os.makedirs(f"/content/benchmark_image/{source}", exist_ok=True)

    for figure in figures:
      for fig_name, img_path in image_path_pairs:
        if figure == fig_name:
          shutil.copy(f"/content/extracted_case_report_image_filtered/{source}/auto/{img_path}", f"/content/benchmark_image/{source}/{figure}.jpg")
          print(f"Image path for {figure}: {img_path}")
  else:
    print(f"{source} question has no image")

  print()

In [ ]:
!zip -r /content/benchmark_image.zip /content/benchmark_image/

In [ ]:
from google.colab import files
files.download('/content/benchmark_image.zip')

# Inspection

In [ ]:
import json

with open('/content/google_gemma-3-4b-it-finetuned_generated_cases_rag.json', 'r') as f:
    data = json.load(f)

print(f"The length of the JSON file is: {len(data)}")

In [ ]:
if 'test_cases' in data:
    test_cases_field = data['test_cases']
    print(f"Type of 'test_cases': {type(test_cases_field)}")
    print(f"Length of 'test_cases': {len(test_cases_field)}")
else:
    print("'test_cases' field not found in the JSON data.")